# Orbit Wars — обучение (SFT) и оценка на Kaggle

Пример полного прогона **на Kaggle**: клонируем код, подключаем данные из
датасета (см. `kaggle_01_publish_data`), коротко обучаем set-transformer
политику и гоняем её в локальном турнире против эвристик.

**Перед запуском:**
1. **Add Input** → подключи датасет `<username>/orbit-wars-sft-data` (из NB1).
2. **Settings → Internet: On** (нужно для `git clone` и `pip install`).
3. **Settings → Accelerator: GPU** (обучение/AMP на CUDA; на CPU будет медленно).


## 1. Клонируем код

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO     = "/kaggle/working/orbit-wars"
REPO_URL = "https://github.com/afarut/orbit-wars.git"
# приватный репозиторий: REPO_URL = "https://<TOKEN>@github.com/afarut/orbit-wars.git"

if not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)

print("repo:", REPO)
print(sorted(os.listdir(REPO))[:20])


## 2. Зависимости

`torch` и `numpy` на Kaggle уже есть. Доставляем Hydra (обучение) и движок
Kaggle + TrueSkill (оценка; тянет тяжёлые транзитивные пакеты — это норм для
офлайна).

In [ ]:
def sh(args, cwd=None):
    print("+", " ".join(str(a) for a in args), f"(cwd={cwd})" if cwd else "")
    return subprocess.run([str(a) for a in args], cwd=cwd, check=True)

# обучение (sft/): Hydra + конфиги
sh([sys.executable, "-m", "pip", "install", "-q",
    "hydra-core==1.3.3", "omegaconf==2.3.1", "tensorboard", "tqdm"])
# оценка (eval/): движок Kaggle (env orbit_wars 1.0.9) + рейтинги
sh([sys.executable, "-m", "pip", "install", "-q",
    "kaggle-environments==1.30.1", "trueskill==0.4.5"])


## 3. Данные → `./data`

Ищем `data.zip` среди подключённых датасетов и распаковываем в корень репо —
появится `<repo>/data/sft.full_send.jsonl` (путь по умолчанию в
`configs/data/full_send.yaml`).

In [ ]:
import glob

cands = glob.glob("/kaggle/input/**/data.zip", recursive=True)
assert cands, "data.zip не найден в /kaggle/input — подключи датасет из NB1 (Add Input)."
DATA_ZIP = cands[0]
print("data.zip:", DATA_ZIP)

sh(["unzip", "-n", "-q", DATA_ZIP, "-d", REPO])   # -n: не перезаписывать (идемпотентно)
print(sorted(os.listdir(Path(REPO) / "data")))
assert (Path(REPO) / "data" / "sft.full_send.jsonl").exists()


## 4. (опц.) Smoke-тест архитектуры

Быстрая проверка форм/масок/инвариантности и валидности `act()` — без движка
Kaggle. Полезно после правок `core/` или `model.py`.

In [ ]:
sh([sys.executable, "smoke_test.py"], cwd=REPO)


## 5. Обучение (SFT)

Короткий демо-прогон: 1 эпоха на срезе ходов (`data.limit`). Устройство
выбирается автоматически (CUDA на Kaggle-GPU, AMP включается сам). Чекпойнты и
TensorBoard пишем в фиксированный `RUN_DIR` (по умолчанию Hydra при
`version_base=None` cwd не меняет — поэтому задаём абсолютные пути явно).

Полный прогон: убери `data.limit`, подними `train.epochs`. Несколько GPU —
`torchrun --standalone --nproc_per_node=N sft/train.py <те же оверрайды>`.

In [ ]:
RUN_DIR = "/kaggle/working/run"

sh([sys.executable, "-m", "sft.train",
    "data=full_send",
    "train.epochs=1",
    "data.limit=50000",            # срез для демо; убери для полного прогона
    "train.batch_size=256",
    "data.num_workers=2",
    f"train.ckpt_dir={RUN_DIR}/checkpoints",
    f"train.tb_dir={RUN_DIR}/tb",
    f"hydra.run.dir={RUN_DIR}/hydra",
], cwd=REPO)


## 6. Где чекпойнты

In [ ]:
ckpt_dir = Path(RUN_DIR) / "checkpoints"
print("ckpt_dir:", ckpt_dir)
print(sorted(p.name for p in ckpt_dir.glob("*.pt")))
best = ckpt_dir / "best.pt"
assert best.exists(), f"нет {best}"
print("best.pt:", best.stat().st_size // 1024, "KB")


## 7. Оценка — локальный турнир

`pool=checkpoints` ставит ростер `[best, bestT, sniper, random]`: наш чекпойнт
(жадный + sampled T=0.7) против эвристик на **настоящем** движке
`make('orbit_wars')`. Числа для демо занижены — для серьёзной оценки подними
`episodes`/`steps` (полная игра = 500 ходов). `device=cpu` + `workers=1`
безопаснее в kernel; инференс быстрый.

In [ ]:
sh([sys.executable, "-m", "eval.run",
    "pool=checkpoints",
    f"ckpt_dir={ckpt_dir}",
    "mode=1v1",
    "episodes=4",                  # карт на матч (для демо мало)
    "steps=200",                   # ходов в эпизоде (полная игра = 500)
    "workers=1",
    "device=cpu",
    f"out={RUN_DIR}/eval",
], cwd=REPO)


## 8. Итоги турнира

In [ ]:
import json

summ = json.loads((Path(RUN_DIR) / "eval" / "summary.json").read_text())
print(f"режим={summ['mode']} игр={summ['n_games']} seed={summ['seed']}\n")
print(f"{'agent':<12}{'skill':>7}{'mu':>7}{'sigma':>7}{'games':>7}{'wins':>6}")
for s in summ["standings"]:
    print(f"{s['label']:<12}{s['skill']:>7.1f}{s['mu']:>7.1f}{s['sigma']:>7.1f}"
          f"{s['games']:>7}{s['wins']:>6}")


## Дальше

- **TensorBoard:** `%load_ext tensorboard` → `%tensorboard --logdir $RUN_DIR/tb`.
- **Полный прогон:** убрать `data.limit`, поднять `train.epochs`; мульти-GPU —
  `torchrun`.
- **Сабмит:** в инференсе участвуют только `model.py` + `core/` + `orbit_lite/`
  (`producer-orbit-wars-utils/`); веса берутся из `best.pt`. `sft/`, `eval/`,
  `dataprep/` в сабмит не входят.
